# Expanded constraint adapter training

**Question:** can the compatible NF4 SDXL workflow cover the expanded training
split within a bounded local run? Read the preregistered protocol before fitting.
Execution, denoising loss and image quality are separate outcomes. All adapters
remain unqualified, inactive by default, with hallucination reduction unverified.

Use the repository venv kernel. Set `HF_HOME` to your existing cached-model
location before starting Jupyter. This notebook performs no provider calls or
backbone downloads. Do not overlap fitting with tests or other model workloads.


In [1]:
from pathlib import Path
import os, sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/training_expanded.yaml').is_file())
sys.path.insert(0, str(root))
os.environ['PYTHON_DOTENV_DISABLED'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
from src.training.common import load_training_config, project_path
from src.training.data import validate_records, pixel_sha256, split_reservations
config = load_training_config(root / 'configs/training_expanded.yaml')
print({'model': config.model, 'resolution': config.resolution, 'steps': config.max_steps, 'quality_status': 'unverified'})

{'model': 'stabilityai/stable-diffusion-xl-base-1.0', 'resolution': 256, 'steps': 320, 'quality_status': 'unverified'}


## 1. Validate attributed data and prior held-out reservations
Only positive train records supervise the generator. I-HallA and false VSR statements are held out from fitting.

In [2]:
records, audit = validate_records(project_path(config.dataset))
prior, _ = split_reservations([root / 'data/training/constraint_pilot/records.jsonl'])
for row in records:
    for key in [row['image_sha256'], pixel_sha256(project_path(config.dataset).parent / row['image'])]:
        if key in prior:
            assert row['split'] == prior[key], 'Prior held-out image assignment changed'
print(audit)
assert sum(row['positive'] and row['split'] == 'train' for row in records) <= config.max_steps * config.gradient_accumulation, 'Planned steps cannot cover the full training split'

{'records': 791, 'positive_by_source_and_split': {'clevr': Counter({'train': 80, 'validation': 10, 'test': 10}), 'vsr': Counter({'train': 240, 'validation': 30, 'test': 30}), 'coco': Counter({'train': 240, 'test': 30, 'validation': 30})}, 'negative_examples': 91, 'dataset_sha256': '6ad7ac828aa6e7554f4c16a32e202c4011cecdd4ff11f4d3b857f022a2ffd8e9', 'image_groups': 700, 'decoded_pixel_groups': 700, 'image_verification_performed': True}


## 2. Check token and orientation compatibility before CUDA
Silent caption truncation, cropping and label-changing flips are prohibited.

In [3]:
from transformers import CLIPTokenizer
from PIL import Image
from src.training.model import training_model_source
from src.training.cache import checked_tokens
source = Path(training_model_source(config.model))
positives = [r for r in records if r['positive'] and r['split'] != 'test']
for folder in ['tokenizer', 'tokenizer_2']:
    tokenizer = CLIPTokenizer.from_pretrained(source / folder, local_files_only=True)
    lengths = []
    for row in positives:
        checked_tokens(tokenizer, row['caption'], row['id'])
        lengths.append(len(tokenizer(row['caption'], truncation=False, add_special_tokens=True)['input_ids']))
    print({'tokenizer': folder, 'captions_checked': len(lengths), 'max_tokens': max(lengths)})
for row in positives:
    with Image.open(project_path(config.dataset).parent / row['image']) as image:
        assert image.getexif().get(274, 1) == 1, 'EXIF orientation requires annotation review'
del tokenizer

E:\BrosKi\framegenx\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'tokenizer': 'tokenizer', 'captions_checked': 630, 'max_tokens': 49}


{'tokenizer': 'tokenizer_2', 'captions_checked': 630, 'max_tokens': 49}


## 3. Inspect the frozen protocol
320 steps, two presentations per step, 256 px, seed 1729. Memory/deadline checks apply at explicit checkpoints, not as OS hard limits. Test images do not enter the frozen cache.

In [4]:
protocol_document = root / 'docs/plans/EXPANDED_TRAINING_PROTOCOL_2026-10-17.md'
print(protocol_document.read_text(encoding='utf-8'))

# Expanded constraint data: preregistered optimization feasibility study

Frozen before new fitting. This is a compatible SDXL PEFT/NF4 study; the prior
unconditional RGB VSM checkpoint is not installed into SDXL. Its negative
0/32-versus-0/32 result remains unchanged.

## Question and limits

Can the existing bounded adapter workflow process a larger, attributed corpus,
cover every training image in a deterministic epoch, preserve held-out images
and finish finite optimization within this laptop's resource budget?

This is an optimization and execution study. It does not establish hallucination
reduction, full-contract CSR, generalization at 512 px or adapter qualification.
The 256 px training resolution reduces compute and preserves full-image geometry
through letterboxing. Its fidelity and inference resolution transfer remain
empirical questions, not assumed improvements.

## Frozen profile

- `configs/training_expanded.yaml`: SDXL base 1.0, NF4 double quantization,
  rank/alpha 4, 

## 4. Fit once or reuse intact evidence
This cell acquires the shared device lease before CUDA. A failed attempt requires explicit inspection/recovery. Identical completed reruns validate hashes and reuse their saved result.

In [5]:
from src.training.study import run_study
result, reused = run_study(config, protocol_document, root / 'runs/training_studies/expanded_2026-10-17')
print({'run_id': result['run_id'], 'status': result['status'], 'reused': reused,
       'steps': result['steps_completed'], 'train_images': result['train_images'],
       'distinct_train_images_seen': result['distinct_train_images_seen'],
       'validation_diagnostic': result['validation_comparison'],
       'host_resource_samples': result['host_resource_samples'],
       'peak_reserved_gb': result['peak_reserved_gb'], 'quality_status': result['quality_status']})

2026/10/17 12:59:10 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at E:\BrosKi\framegenx\venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


W1017 12:59:14.468000 22644 venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 2941.75it/s]

Cached text_encoder for 630 positive captions


...

step 105/320 mse=0.141127 vram=2.307GB elapsed=1783.5s


step 106/320 mse=0.331597 vram=2.307GB elapsed=1811.5s


Traceback (most recent call last):
  File "E:\BrosKi\framegenx\src\training\train.py", line 280, in run_training
    raise TimeoutError("Training exceeded its configured wall-clock budget")
TimeoutError: Training exceeded its configured wall-clock budget


{
  "run_id": "77361340-f9b7-4319-8392-f2d6fefe8842",
  "status": "failed",
  "quality_status": "unverified",
  "promotion_status": "unqualified",
  "metric_scope": "training_objective_only",
  "steps_completed": 106,
  "started_at": "2026-10-17T07:29:02.049332+00:00",
  "config": {
    "model": "stabilityai/stable-diffusion-xl-base-1.0",
    "method": "qlora",
    "dataset": "data/training/constraint_expanded_v1/records.jsonl",
    "output_root": "runs/training",
    "tracking_db": "runs/mlflow.db",
    "experiment": "framegenx-expanded-constraint-feasibility",
    "resolution": 256,
    "rank": 4,
    "lora_alpha": 4,
    "learning_rate": 1e-05,
    "max_steps": 320,
    "gradient_accumulation": 2,
    "seed": 1729,
    "deterministic": true,
    "cpu_threads": 4,
    "checkpoint_every": 80,
    "validation_every": 80,
    "validation_limit": 12,
    "max_vram_gb": 5.5,
    "min_free_vram_gb": 3.0,
    "max_runtime_seconds": 1800,
    "max_total_runtime_seconds": 2700,
    "max_proce

ValueError: Study did not complete its planned fitting and image coverage

## 5. Interpret the actual result
Full image coverage and an intact checkpoint verify this optimization workflow. Lower denoising loss does not establish better generated counts, object/color binding or full-contract CSR. A separate paired image study with reviewed, disjoint labels must qualify the candidate. The failed scratch VSM Cards study is retained.

In [ ]:
assert result['promotion_status'] == 'unqualified'
assert result['quality_status'] == 'unverified'
print({'automatic_activation': False, 'hallucination_reduction': 'unverified', 'next_gate': 'disjoint paired image evaluation and human-calibrated requirement evidence'})